# Experimenting with hybrid search in out Qdrant DB

In [38]:
from dotenv import load_dotenv
load_dotenv("../../.env")

import openai
import pandas as pd

from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, RrfQuery, Rrf

### New Collection for hybrid search

In [2]:
qdrant_client = QdrantClient(url="localhost:6333")

In [4]:
from numpy import size
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-hybrid-search",
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE),
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams(modifier=Modifier.IDF),
    }
)

True

In [5]:
qdrant_client.create_payload_index(
    collection_name="Amazon-items-collection-hybrid-search",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding Functions

In [7]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]

    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend(
            [embedding.embedding for embedding in response.data]
        )

        print(f"Batch #{counter}: {counter * batch_size} / {len(text_list)}")
        counter += 1

    return all_embeddings

### Read the sampled dataset with Amazon inventory data

In [8]:
df_items = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [9]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,AMAZON FASHION,"Bosttor Bluetooth Beanie Hat with Light, Headl...",4.5,2342,"[100% Acrylic, Elastic closure, Hand Wash Only...",[],26.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Beanie Hat with Bluetooth Headphon...,Bosttor,"[Electronics, Headphones, Earbuds & Accessorie...","{'Department': 'unisex-adult', 'Date First Ava...",B0BH41HYFZ,NaN,NaN,NaN
1,All Electronics,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",4.2,148,[【USB or USB C to Ethernet Adapter】The Etherne...,[],14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aceele,"[Electronics, Computers & Accessories, Network...",{'Product Dimensions': '3.54 x 0.98 x 0.71 inc...,B0BKPB2YQ9,NaN,NaN,NaN
2,All Electronics,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,4.2,3331,[📍3-Port HDMI Switch: This aluminum HDMI switc...,[],18.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Darren reviews VWRHAR 3-1 splitter...,VWRHar,"[Electronics, Home Audio, Home Audio Accessori...",{'Package Dimensions': '6.1 x 4.21 x 0.94 inch...,B09MM5QT3R,NaN,NaN,NaN
3,All Electronics,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",3.4,125,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Burxoe,[],{'Package Dimensions': '3.15 x 3.07 x 2.52 inc...,B09Q5TNDHY,NaN,NaN,NaN
4,Cell Phones & Accessories,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",4.7,335,[【Compatible Airpods 1/2 】This case designed f...,[1],11.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'airpod cases', 'url': 'https://www...",UGUHY,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '7.44 x 4.61 x 1.57 inc...,B0BZJKX7MS,NaN,NaN,NaN


### Preprocess title and features

In [10]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [11]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")

In [12]:
df_items["preprocessed_description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [13]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,preprocessed_description,image
0,AMAZON FASHION,"Bosttor Bluetooth Beanie Hat with Light, Headl...",4.5,2342,"[100% Acrylic, Elastic closure, Hand Wash Only...",[],26.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Beanie Hat with Bluetooth Headphon...,Bosttor,"[Electronics, Headphones, Earbuds & Accessorie...","{'Department': 'unisex-adult', 'Date First Ava...",B0BH41HYFZ,NaN,NaN,NaN,"Bosttor Bluetooth Beanie Hat with Light, Headl...",https://m.media-amazon.com/images/I/51b7qcj8dZ...
1,All Electronics,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",4.2,148,[【USB or USB C to Ethernet Adapter】The Etherne...,[],14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aceele,"[Electronics, Computers & Accessories, Network...",{'Product Dimensions': '3.54 x 0.98 x 0.71 inc...,B0BKPB2YQ9,NaN,NaN,NaN,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",https://m.media-amazon.com/images/I/41WsGRr-3T...
2,All Electronics,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,4.2,3331,[📍3-Port HDMI Switch: This aluminum HDMI switc...,[],18.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Darren reviews VWRHAR 3-1 splitter...,VWRHar,"[Electronics, Home Audio, Home Audio Accessori...",{'Package Dimensions': '6.1 x 4.21 x 0.94 inch...,B09MM5QT3R,NaN,NaN,NaN,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,https://m.media-amazon.com/images/I/41fsjaknf1...
3,All Electronics,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",3.4,125,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Burxoe,[],{'Package Dimensions': '3.15 x 3.07 x 2.52 inc...,B09Q5TNDHY,NaN,NaN,NaN,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",https://m.media-amazon.com/images/I/51YFypeuZh...
4,Cell Phones & Accessories,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",4.7,335,[【Compatible Airpods 1/2 】This case designed f...,[1],11.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'airpod cases', 'url': 'https://www...",UGUHY,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '7.44 x 4.61 x 1.57 inc...,B0BZJKX7MS,NaN,NaN,NaN,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",https://m.media-amazon.com/images/I/41V7Wi2ckb...


In [14]:
list(df_items["preprocessed_description"].items())[0]

(0,
 'Bosttor Bluetooth Beanie Hat with Light, Headlamp Cap with Headphones and Built-in Speaker Mic, Gifts for Men Women Teen 100% Acrylic Elastic closure Hand Wash Only 【Upgraded Bluetooth Beanie】Bosttor upgraded Bluetooth wireless technology offer much stable and strong connection, support music and calling, easy and fast to pair with your devices. Maximum transmission distance up to 33 feet. Built-in Stereo Speakers & Mic, enhance your music listening experience. 【Long Working Time Music Hat】Built with easy-access USB charging port, simply charge it via the included USD cable. It takes 1-2 hours to get full charged. The battery offers continuous working hours up to 10 hours. Perfect for camping, hiking, skiing, hunting, jogging, cycling, dog walking, auto repair, etc. 【LED Headlight Light Up Your Way】Hands-free LED light is rechargeable and removable. You can remove it to charge by laptop, power bank, socket, car charger, etc. The 4 LED lights can light up to 30 feet away. It point

In [15]:
list(df_items["image"].items())[0]

(0, 'https://m.media-amazon.com/images/I/51b7qcj8dZL._AC_.jpg')

### Sample 50 items from the dataset

In [16]:
df_sample = df_items.sample(n=50, random_state=42)

In [17]:
df_data_to_embed = df_sample[["preprocessed_description", "image", "rating_number", "price", "average_rating", "parent_asin"]]

In [18]:
df_data_to_embed.head()

,preprocessed_description,image,rating_number,price,average_rating,parent_asin
521,Marame 120mm 5v USB Powered Fan with Speed Con...,https://m.media-amazon.com/images/I/41Zpg4vSEj...,104,14.99,4.7,B0BRJS644Z
737,"Kids Wireless Headphones, Adjustable Headband,...",https://m.media-amazon.com/images/I/41+jaEbUbY...,173,NaN,4.2,B09KQP2H7N
740,"Veetone Aux Cord for iPhone, 3.3ft [Apple MFi ...",https://m.media-amazon.com/images/I/31ag2Blvjm...,2218,5.99,4.0,B0CC4HBS85
660,AICHESON Laptop Cooler Pad with 6 Cooling Fans...,https://m.media-amazon.com/images/I/51BKtk3-RD...,145,NaN,4.2,B099N9F3FP
411,"Raymate Bluetooth Speakers, HiFi Stereo Sound ...",https://m.media-amazon.com/images/I/31SnPdXS77...,270,38.69,4.7,B0C996WY16


In [19]:
data_to_embed = df_data_to_embed.to_dict(orient="records")

In [20]:
data_to_embed

[{'preprocessed_description': 'Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】\xa0Rubber Feet on the fan raise it up enough off of the surface it is sittin

In [21]:
len(data_to_embed)

50

In [23]:
text_for_embedding = [item["preprocessed_description"] for item in data_to_embed]

In [24]:
text_for_embedding

['Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】\xa0Rubber Feet on the fan raise it up enough off of the surface it is sitting on to allow enough replacem

In [25]:
embeddings = get_embeddings_batch(text_for_embedding)

In [26]:
len(embeddings)

50

In [27]:
pointstructs = []
counter = 1

for embedding, data in zip(embeddings, data_to_embed):
    pointstructs.append(
        PointStruct(
            id=counter,
            vector={
                "text-embedding-3-small": embedding,
                "bm25": Document(
                    text=data["preprocessed_description"],
                    model="qdrant/bm25"
                )
            },
            payload=data
        )
    )
    counter += 1

In [28]:
pointstructs[0].vector

{'text-embedding-3-small': [0.035614013671875,
  0.0220947265625,
  -0.041412353515625,
  -0.0002410411834716797,
  -0.0084381103515625,
  -0.005153656005859375,
  -0.01128387451171875,
  -0.029296875,
  0.06805419921875,
  -0.01239013671875,
  0.00494384765625,
  0.00739288330078125,
  -0.06622314453125,
  0.045867919921875,
  0.0045318603515625,
  -0.01537322998046875,
  -0.02276611328125,
  0.005382537841796875,
  -0.0194091796875,
  0.0284423828125,
  0.03558349609375,
  0.0303192138671875,
  -0.0078582763671875,
  -0.00029206275939941406,
  -0.0006275177001953125,
  -0.002140045166015625,
  0.031829833984375,
  0.0513916015625,
  0.022369384765625,
  -0.043792724609375,
  -0.032196044921875,
  -0.0266876220703125,
  -0.02093505859375,
  -0.042510986328125,
  -0.0297698974609375,
  0.00405120849609375,
  -0.0305938720703125,
  -0.05157470703125,
  -0.01238250732421875,
  -0.0058746337890625,
  0.01904296875,
  -0.047576904296875,
  0.0230865478515625,
  0.02239990234375,
  0.029373

In [29]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-hybrid-search",
    points=pointstructs,
    wait=True
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

### Hybrid Search

In [39]:
def retrieve_data_hybrid(query, k=10):
    query_embedding = get_embeddings_batch(query)[0]

    #hybrid retrieval
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=k
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=k
            )
        ],
        query=RrfQuery(rrf=Rrf(weights=[0.4, 0.6])),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


In [41]:
results = retrieve_data_hybrid("What are some available tablets?", k=20)

In [42]:
results

{'retrieved_context_ids': ['B0BG6TMXDD',
  'B0BBF2VC6X',
  'B0C9ZWCZ99',
  'B0B5LW6277',
  'B0BYRWPV86',
  'B0C9QZS95R',
  'B09WCT9S1R',
  'B0CF57H28T',
  'B0C8S6BBY9',
  'B0BM9THPDQ',
  'B09QGRRY7G',
  'B0CH8DRD6K',
  'B09YHG2Z7F',
  'B09P4QW5Y2',
  'B09KQP2H7N',
  'B09PTX6461',
  'B0BB6TFQ22',
  'B09V2Y1TQB',
  'B0BM657X74',
  'B0BMFYWK6T'],
 'retrieved_context': ['Fire HD 8 & HD 8 Plus Tablet Case for Kids (Only 12th Gen, 2022 Release) - DJ&RPPQ Lightweight Shockproof Cover with Handle Stand for Kindle Fire HD 8 Kids Tablet & Kids Pro Tablet - Blue Designed for Amazon All-New Kindle Fire HD 8 & Fire HD 8 Plus Tablet, Fire HD 8 Kids tablet & Fire HD 8 Kids Pro tablet (12th Generation 2022 Release), NOT fit for other models. Cutouts for clear access to all Amazon Kindle Fire HD 8 Plus / Fire HD 8 Tablet(12TH Generation, 2022Release) / Fire HD 8 Kids Pro 2022Tablet buttons, ports, speakers and rear-camera. Raised screen bezel edges for extra protection when fall. Kindle fire 8 case mad